# L'écosystème pandas, polars, DuckDB, Spark

Ce notebook ne vous apprend pas polars, DuckDB ou Spark.
Il vous donne le **pied à l'étrier** : savoir ce qui existe, quand l'utiliser,
et à quoi ça ressemble pour ne pas être déstabilisé le jour où vous en croisez un.

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
from _data import load_accidents

try:
    import polars as pl
    POLARS_DISPONIBLE = True
    print(f"polars {pl.__version__} disponible")
except ImportError:
    POLARS_DISPONIBLE = False
    print("polars non installé — les sections polars afficheront ce message.")
    print("Pour installer : uv add polars")

try:
    import duckdb
    DUCKDB_DISPONIBLE = True
    print(f"duckdb {duckdb.__version__} disponible")
except ImportError:
    DUCKDB_DISPONIBLE = False
    print("duckdb non installé : uv add duckdb")

## 1. Préparation

On sauvegarde le dataset accidents en Parquet — c'est le format d'échange neutre
entre tous les outils qui suivent. Chacun peut lire un fichier Parquet sans connaître
les autres.

In [ ]:
df = load_accidents()
df.to_parquet("accidents.parquet")
print(f"{len(df):,} lignes, {df.shape[1]} colonnes — accidents.parquet écrit.")

**La question de référence** pour tous les exemples qui suivent :

> *Top 10 des départements par nombre d'accidents, sur les accidents de nuit (lum >= 3).*

Même question, même fichier, cinq outils.

## 2. Panorama en une image

Cette cellule se **lit**, elle ne s'exécute pas.
En 8 minutes au tableau, l'animateur peut la parcourir à voix haute
et les participants voient à l'œil nu les similitudes et différences de philosophie.

***

| Outil | Répondre à la question | Nature |
|---|---|---|
| **pandas** | `pd.read_parquet("accidents.parquet").query("lum >= 3").groupby("dep")["Num_Acc"].count().nlargest(10)` | In-memory, eager, interactif |
| **polars** (eager) | `pl.read_parquet("accidents.parquet").filter(pl.col("lum") >= 3).group_by("dep").agg(pl.len().alias("nb")).sort("nb", descending=True).head(10)` | In-memory, eager, Rust |
| **polars** (lazy) | `pl.scan_parquet("accidents.parquet").filter(pl.col("lum") >= 3).group_by("dep").agg(pl.len().alias("nb")).sort("nb", descending=True).head(10).collect()` | Lazy — ne lit que ce dont il a besoin |
| **DuckDB** | `duckdb.sql("SELECT dep, COUNT(*) nb FROM 'accidents.parquet' WHERE lum >= 3 GROUP BY dep ORDER BY nb DESC LIMIT 10").df()` | SQL, vectorisé, embarqué |
| **PySpark** | `spark.read.parquet("accidents.parquet").filter(F.col("lum") >= 3).groupBy("dep").agg(F.count("*").alias("nb")).orderBy(F.desc("nb")).limit(10).show()` | Distribué (JVM + cluster) |
| **dask** | `dd.read_parquet("accidents.parquet").query("lum >= 3").groupby("dep")["Num_Acc"].count().nlargest(10).compute()` | pandas-like, parallèle |
| **ibis** | `ibis.read_parquet("accidents.parquet").filter(_.lum >= 3).group_by("dep").aggregate(nb=_.Num_Acc.count()).order_by(ibis.desc("nb")).limit(10).to_pandas()` | DSL unifié, backend swappable |

> **PySpark, dask et ibis** nécessitent de l'infrastructure supplémentaire
> (JVM pour Spark, configuration de cluster ou worker pour dask en distribué).
> Les cellules de ce notebook n'exécutent que pandas, polars et DuckDB.

***

**Ce qu'on observe à l'œil nu :**
- pandas et polars se ressemblent beaucoup — method chaining, mêmes concepts (filter/query, group_by/groupby, sort/nlargest)
- DuckDB, c'est du SQL pur — si vous pensez déjà en SQL, il n'y a rien à apprendre
- PySpark ressemble à polars mais les transformations sont déclaratives sur un DAG distribué
- ibis est le plus abstrait : même syntaxe quel que soit le backend sous-jacent

## 3. Zoom sur polars — l'alternative la plus proche de pandas

Parmi tous ces outils, polars est celui dont l'API est la plus proche du pandas moderne.
Même style chaîné, mêmes verbes (`filter`, `group_by`, `agg`, `sort`), mais :

- Moteur écrit en **Rust** — 5 à 20x plus rapide que pandas sur les mêmes opérations
- **Lazy par défaut** : polars optimise le plan d'exécution avant de lire quoi que ce soit
- **Pas d'index** : polars a supprimé l'index pandas — toutes les colonnes sont égales
- **Expressions explicites** : `pl.col("dep")` au lieu de `df["dep"]` — plus verbeux mais sans ambiguïté

Si un jour pandas rame sur vos données, polars est le premier réflexe à avoir.
Pas de JVM, pas de cluster, `uv add polars` et c'est parti.

## 4. Premier exemple polars — mode éager

In [ ]:
if not POLARS_DISPONIBLE:
    print("polars non installé — cellule ignorée.")
else:
    resultat_eager = (
        pl.read_parquet("accidents.parquet")           # charge tout en mémoire
        .filter(pl.col("lum") >= 3)                    # filtre sur luminosité (nuit)
        .group_by("dep")                               # grouper par département
        .agg(pl.len().alias("nb_accidents"))           # compter les accidents
        .sort("nb_accidents", descending=True)         # trier décroissant
        .head(10)                                      # top 10
    )
    print(resultat_eager)

**Différences de syntaxe à noter par rapport à pandas :**

| pandas | polars |
|---|---|
| `df["col"]` | `pl.col("col")` |
| `.query("lum >= 3")` | `.filter(pl.col("lum") >= 3)` |
| `.groupby("dep")` | `.group_by("dep")` |
| `.agg(nb=("Num_Acc", "count"))` | `.agg(pl.len().alias("nb"))` |
| `.nlargest(10, "nb")` | `.sort("nb", descending=True).head(10)` |
| `df["col"].str.contains("x")` | `pl.col("col").str.contains("x")` |

## 5. Deuxième exemple polars — mode lazy

En mode lazy, polars ne lit et ne calcule que ce dont il a besoin.
Si la requête n'utilise que 2 colonnes et filtre 90 % des lignes,
polars ne lira jamais les autres colonnes ni les lignes filtrées.

La différence avec pandas : **pandas charge tout en mémoire d'abord, puis transforme**.
polars lazy **décrit d'abord la transformation, puis décide quoi charger**.

In [ ]:
if not POLARS_DISPONIBLE:
    print("polars non installé — cellule ignorée.")
else:
    # scan_parquet au lieu de read_parquet — rien n'est chargé ici
    plan = (
        pl.scan_parquet("accidents.parquet")           # plan, pas de lecture
        .filter(pl.col("lum") >= 3)
        .group_by("dep")
        .agg(pl.len().alias("nb_accidents"))
        .sort("nb_accidents", descending=True)
        .head(10)
    )

    # Afficher le plan d'exécution optimisé — avant de lire quoi que ce soit
    print(plan.explain())

In [ ]:
if not POLARS_DISPONIBLE:
    print("polars non installé — cellule ignorée.")
else:
    # .collect() déclenche l'exécution — lecture + calcul en une passe optimisée
    resultat_lazy = plan.collect()
    print(resultat_lazy)

## 6. DuckDB — SQL directement sur le fichier Parquet

In [ ]:
if not DUCKDB_DISPONIBLE:
    print("duckdb non installé — cellule ignorée.")
else:
    resultat_duckdb = duckdb.sql("""
        SELECT dep, COUNT(*) AS nb_accidents
        FROM 'accidents.parquet'
        WHERE lum >= 3
        GROUP BY dep
        ORDER BY nb_accidents DESC
        LIMIT 10
    """).df()  # .df() convertit en pandas DataFrame

    print(resultat_duckdb)

DuckDB peut aussi requêter **directement une DataFrame pandas** en mémoire — sans passer par un fichier.

In [ ]:
if not DUCKDB_DISPONIBLE:
    print("duckdb non installé — cellule ignorée.")
else:
    # df est la DataFrame pandas chargée plus haut
    # DuckDB peut la lire directement par son nom de variable
    duckdb.sql("""
        SELECT dep, COUNT(*) AS nb_accidents
        FROM df
        WHERE lum >= 3
        GROUP BY dep
        ORDER BY nb_accidents DESC
        LIMIT 5
    """).df()

## 7. Comparaison de performance

In [ ]:
%%timeit
# pandas — depuis le fichier Parquet
(
    pd.read_parquet("accidents.parquet")
    .query("lum >= 3")
    .groupby("dep")["Num_Acc"]
    .count()
    .nlargest(10)
)

In [ ]:
if not POLARS_DISPONIBLE:
    print("polars non installé.")
else:
    get_ipython().run_cell_magic("timeit", "", """
pl.read_parquet("accidents.parquet").filter(pl.col("lum") >= 3).group_by("dep").agg(pl.len().alias("nb")).sort("nb", descending=True).head(10)
""")

In [ ]:
if not POLARS_DISPONIBLE:
    print("polars non installé.")
else:
    get_ipython().run_cell_magic("timeit", "", """
pl.scan_parquet("accidents.parquet").filter(pl.col("lum") >= 3).group_by("dep").agg(pl.len().alias("nb")).sort("nb", descending=True).head(10).collect()
""")

> Sur ce dataset (~60 000 lignes), la différence est modeste — pandas est déjà rapide en dessous de quelques millions de lignes.
> polars prend un avantage net à partir de **~1 Go de données** et devient indispensable au-delà de **10 Go**.
> Le mode lazy est surtout utile quand le filtre réduit drastiquement la quantité de données lues.

## 8. Différence de philosophie en une phrase

- **pandas** = *"je charge toutes mes données en mémoire, puis j'explore interactivement."*
- **polars** = *"je décris une requête complète, polars décide quoi charger et dans quel ordre."*
- **DuckDB** = *"j'écris du SQL, DuckDB exécute comme un moteur de base de données sans serveur."*
- **PySpark** = *"je décris un graphe de transformations, le cluster exécute en parallèle sur N machines."*

## 9. Ce qu'il faut retenir

| Situation | Outil recommandé |
|---|---|
| Exploration quotidienne, données < 1 Go | **pandas** |
| pandas commence à ramer (1–50 Go, une machine) | **polars** |
| Vous pensez en SQL, données en Parquet/CSV | **DuckDB** |
| Données > 100 Go, cluster disponible | **PySpark** |
| Même code Python, plusieurs backends | **ibis** |
| pandas-like, parallélisme sans cluster | **dask** |

**Règle pratique** : commencez par pandas. Migrez quand vous avez un problème concret
(mémoire insuffisante, temps de calcul trop long, données distribuées).
Ne choisissez pas un outil par anticipation — choisissez-le quand pandas ne suffit plus.

## Quiz d'orientation

<details><summary>Vous avez 800 Go de logs sur un laptop avec 32 Go de RAM. Quel outil ?</summary>

**DuckDB** ou **polars lazy**. DuckDB peut lire des fichiers Parquet par colonnes et par blocs
sans jamais charger le fichier entier. polars en mode lazy fait de même avec `scan_parquet()`.
pandas est exclu : il chargerait tout en RAM d'un coup.

</details>

<details><summary>Votre collègue vous envoie un script polars. Vous savez pandas moderne. Pouvez-vous le lire ?</summary>

Oui, très largement. Les concepts sont les mêmes (filter, group_by, agg, sort, join).
La principale différence syntaxique est `pl.col("nom")` à la place de `df["nom"]` dans les expressions.
Tout le reste est du method chaining que vous reconnaissez.

</details>

In [ ]:
# Nettoyage
import os
if os.path.exists("accidents.parquet"):
    os.remove("accidents.parquet")